# Day 16 · 数据摄取
## 今日学习目标
- 掌握 CSV / Excel / JSON 读取
- 掌握 SQL 数据库读取
- 掌握 API 数据拉取与数据导出

## 引入
思考：`plt.hist(df['年龄'])` 画的是什么图？
- 答案：直方图（histogram），用来观察数值分布。

但 `df` 从哪来？今天学习的就是：**如何把外部数据变成 DataFrame**。
数据常见来源：
1. CSV 文件（最常见）
2. Excel 报表
3. JSON 接口 / 文件
4. SQL 数据库
5. HTTP API

## 第一讲：CSV 与 Excel 读取

### 1.1 read_csv 参数详解

In [1]:
import pandas as pd
from io import StringIO

# 模拟 CSV 字符串（实际用 pd.read_csv('data.csv')）
csv_data = """姓名,年龄,城市
张三,25,北京
李四,30,上海
王五,28,深圳
赵六,35,广州"""

# read_csv 常用参数：
#   sep=','        分隔符，默认逗号
#   encoding='utf-8'  编码（中文常用 gbk / utf-8）
#   index_col=0    把第 0 列当作索引
#   usecols=['姓名','年龄']  只读指定列
#   nrows=10       只读前 10 行
#   skiprows=2     跳过前 2 行
#   na_values=['NA','']  把哪些值当作缺失值
df = pd.read_csv(StringIO(csv_data))
print("=== 原始数据 ===")
print(df.head())

# 演示：跳过前 2 行，只选两列
df2 = pd.read_csv(
    StringIO(csv_data),
    skiprows=2,
    usecols=['姓名', '年龄']
)
print("\n=== 跳过前 2 行，只用姓名和年龄列 ===")
print(df2)
print(df2.info())

=== 原始数据 ===
   姓名  年龄 城市
0  张三  25  北京
1  李四  30  上海
2  王五  28  深圳
3  赵六  35  广州

=== 跳过前 2 行，只用姓名和年龄列 ===
   姓名  年龄
0  王五  28
1  赵六  35

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   姓名    2 non-null      object
 1   年龄    2 non-null      int64 
dtypes: int64(1), object(1)
memory usage: 160.0+ bytes
None


> 🔴 易错点：中文 CSV 文件多数用 `gbk` 编码，
而不是 `utf-8`。乱码时先尝试：
`pd.read_csv('file.csv', encoding='gbk')`

### 1.2 read_excel

In [2]:
import pandas as pd
from io import BytesIO

# read_excel 常用参数：
#   sheet_name=0        第 1 个 sheet（int）
#   sheet_name='Sheet2' 按名字取
#   sheet_name=None     全部 sheet，返回 {name: df} 字典
#   usecols / header    用法同 read_csv
print("=== read_excel 常用参数（注释） ===")
print("# pd.read_excel('data.xlsx', sheet_name='Sheet2')")
print("# pd.read_excel('data.xlsx', sheet_name=None)"
print("  读全部 sheet")
print("# pd.read_excel('data.xlsx', usecols=[0, 1, 2])")
print("# pd.read_excel('data.xlsx', header=1)")

# 模拟：构造一个含两个 sheet 的 Excel 文件
with pd.ExcelWriter('demo.xlsx') as writer:
    pd.DataFrame({'姓名': ['张三', '李四'], '成绩': [88, 92]}).to_excel(
        writer, sheet_name='数学', index=False
    )
    pd.DataFrame({'姓名': ['张三', '李四'], '成绩': [76, 85]}).to_excel(
        writer, sheet_name='语文', index=False
    )

all_sheets = pd.read_excel('demo.xlsx', sheet_name=None)
print("\n=== 模拟多 sheet 读取 ===")
print(all_sheets)

import os
os.remove('demo.xlsx')

=== read_excel 常用参数（注释） ===
# pd.read_excel('data.xlsx', sheet_name='Sheet2')
# pd.read_excel('data.xlsx', sheet_name=None)  读全部 sheet
# pd.read_excel('data.xlsx', usecols=[0, 1, 2])
# pd.read_excel('data.xlsx', header=1)

=== 模拟多 sheet 读取 ===
{'数学':   姓名  成绩
0  张三   88
1  李四   92, '语文':   姓名  成绩
0  张三  76
1  李四  85}


### ✏️ 练习 1

In [ ]:
import pandas as pd
from io import StringIO

# 含中文的 CSV 字符串（第 3 行才是数据）
csv_data = """这是说明行
第二行也是标题
姓名,年龄,城市
张三,25,北京
李四,30,上海"""

# 练习：用 read_csv 读取，encoding='gbk'，跳过前 2 行
# 学员代码区
pass

In [3]:
# 参考答案
import pandas as pd
from io import StringIO

csv_data = """这是说明行
第二行也是标题
姓名,年龄,城市
张三,25,北京
李四,30,上海"""

csv_file = StringIO(csv_data)
df = pd.read_csv(csv_file, skiprows=2)
print(df)

   姓名  年龄  城市
0  张三  25  北京
1  李四  30  上海


In [ ]:
# 练习：读取 demo.xlsx 里的 sheet '语文'，只取姓名和成绩
# 学员代码区
pass

In [4]:
# 参考答案
import pandas as pd

# 先构造一个临时 Excel 并删掉
with pd.ExcelWriter('demo.xlsx') as writer:
    pd.DataFrame(
        {'姓名': ['张三', '李四'], '成绩': [88, 92]}
    ).to_excel(writer, sheet_name='数学', index=False)
    pd.DataFrame(
        {'姓名': ['张三', '李四'], '成绩': [76, 85]}
    ).to_excel(writer, sheet_name='语文', index=False)

df = pd.read_excel(
    'demo.xlsx',
    sheet_name='语文',
    usecols=['姓名', '成绩']
)
print(df)

import os
os.remove('demo.xlsx')

   姓名  成绩
0  张三  76
1  李四  85


## 第二讲：JSON 与数据库读取

### 2.1 read_json

In [5]:
import pandas as pd
import json

# JSON 字符串
json_str = '''[
    {"姓名": "张三", "年龄": 25, "城市": "北京"},
    {"姓名": "李四", "年龄": 30, "城市": "上海"},
    {"姓名": "王五", "年龄": 28, "城市": "深圳"}
]'''

# read_json 常用参数：
#   orient='records'  JSON 数组格式
#   force_ascii=False 中文不转义
#   lines=True         每行一条记录
df = pd.read_json(json_str)
print("=== read_json：从 JSON 字符串读入 ===")
print(df)

=== read_json：从 JSON 字符串读入 ===
   姓名  年龄 城市
0  张三  25  北京
1  李四  30  上海
2  王五  28  深圳


### 2.2 read_sql

In [6]:
import sqlite3
import pandas as pd

# sqlite3.connect('mydb.db')  # 真实场景连接文件
# 这里用 :memory: 内存数据库做演示

# 用 with 自动关闭连接
with sqlite3.connect(':memory:') as conn:
    cursor = conn.cursor()
    cursor.execute(
        'CREATE TABLE students '
        '(id INTEGER PRIMARY KEY, 姓名 TEXT, 年龄 INTEGER)'
    )
    cursor.executemany(
        'INSERT INTO students (姓名, 年龄) VALUES (?, ?)',
        [('张三', 25), ('李四', 30), ('王五', 28)]
    )
    conn.commit()

    # pd.read_sql 读整张表
    df_all = pd.read_sql('SELECT * FROM students', conn)
    print("=== 用 with 上下文管理器读取 SQLite ===")
    print(df_all)

    # 带条件查询
    df_filtered = pd.read_sql(
        'SELECT * FROM students WHERE 年龄 >= 30', conn
    )
    print("\n=== 用 SQL 查询过滤 ===")
    print(df_filtered)
# with 结束，连接自动关闭

=== 用 with 上下文管理器读取 SQLite ===
   id  姓名  年龄
0   1  张三  25
1   2  李四  30
2   3  王五  28

=== 用 SQL 查询过滤 ===
   id  姓名  年龄
0   2  李四  30
1   3  王五  28


> 🔴 易错点：忘记关闭数据库连接会导致文件锁死。
建议始终使用 `with conn:` 或 `with sqlite3.connect(...) as conn:`

### ✏️ 练习 2

In [ ]:
import pandas as pd

# 嵌套 JSON：每个学生有个 'scores' 字段（dict）
json_str = '''[
    {"姓名": "张三", "scores": {"数学": 88, "语文": 76}},
    {"姓名": "李四", "scores": {"数学": 92, "语文": 85}}
]'''

# 练习：读入并提取每个人的数学成绩到新列
# 学员代码区
pass

In [7]:
# 参考答案
import pandas as pd

json_str = '''[
    {"姓名": "张三", "scores": {"数学": 88, "语文": 76}},
    {"姓名": "李四", "scores": {"数学": 92, "语文": 85}}
]'''

df = pd.read_json(json_str)
df['数学'] = df['scores'].apply(lambda x: x['数学'])
df = df[['姓名', '数学']]
print(df)

   姓名  数学
0  张三  88
1  李四  92


In [ ]:
import sqlite3
import pandas as pd

# 练习：用 read_sql 从 SQLite 查询，
#       用 with 自动关闭连接
# 学员代码区
pass

In [8]:
# 参考答案
import sqlite3
import pandas as pd

with sqlite3.connect(':memory:') as conn:
    cursor = conn.cursor()
    cursor.execute(
        'CREATE TABLE users '
        '(id INTEGER PRIMARY KEY, 姓名 TEXT, 年龄 INTEGER)'
    )
    cursor.executemany(
        'INSERT INTO users (姓名, 年龄) VALUES (?, ?)',
        [('张三', 25), ('李四', 30)]
    )
    conn.commit()
    df = pd.read_sql('SELECT * FROM users', conn)
    print(df)

   id  姓名  年龄
0   1  张三  25
1   2  李四  30


## 第三讲：API 数据拉取与数据导出

### 3.1 从 API 拉取数据

In [9]:
import requests
import pandas as pd

# 请求公开测试 API
url = 'https://jsonplaceholder.typicode.com/users'
response = requests.get(url)

if response.status_code == 200:
    data = response.json()          # 得到 list[dict]
    df = pd.DataFrame(data)
    print("=== 请求 JSONPlaceholder 用户接口 ===")
    print(df[['id', 'name', 'email']].head(3))
    print(f"\n返回了 {len(df)} 条用户数据")
else:
    print(f"请求失败: {response.status_code}")

=== 请求 JSONPlaceholder 用户接口 ===
   id     name                 email
0   1    Leanne Graham     Sincere@april.biz
1   2    Ervin Howell      Shanna@melissa.tv
2   3  Clementine Bauch  Nathan@yesenia.net

返回了 10 条用户数据


### 3.2 数据导出

In [10]:
import pandas as pd
import sqlite3
import os

df = pd.DataFrame({
    '姓名': ['张三', '李四', '王五'],
    '年龄': [25, 30, 28],
    '城市': ['北京', '上海', '深圳']
})

# to_csv：index=False 避免多出一列序号
df.to_csv('out.csv', index=False, encoding='utf-8')
print("=== 导出为 CSV（第一行预览）===")
print(open('out.csv', encoding='utf-8').read().split('\n')[0])
print(open('out.csv', encoding='utf-8').read().split('\n')[1])

# to_json：orient='records', force_ascii=False 保留中文
json_out = df.to_json(
    orient='records', force_ascii=False
)
print("\n=== 导出为 JSON（orient='records'）===")
print(json_out[:60])

# to_sql：if_exists='replace' 表已存在则替换
with sqlite3.connect('out.db') as conn:
    df.to_sql('people', conn, if_exists='replace', index=False)
    count = pd.read_sql('SELECT COUNT(*) FROM people', conn)
    print("\n=== 导出到 SQLite ===")
    print(f"导出成功，共 {len(df)} 行")

# 清理临时文件
os.remove('out.csv')
os.remove('out.db')

=== 导出为 CSV（第一行预览）===
姓名,年龄,城市
张三,25,北京

=== 导出为 JSON（orient='records'）===
[{"姓名":"张三","年龄":25,"城市":"北京"}]

=== 导出到 SQLite ===
导出成功，共 3 行


### ✏️ 练习 3

In [ ]:
import requests
import pandas as pd

# 练习：调用 JSONPlaceholder 的 /posts 接口
#       把返回的 JSON 转为 DataFrame，打印前 5 行
# 学员代码区
pass

In [11]:
# 参考答案
import requests
import pandas as pd

url = 'https://jsonplaceholder.typicode.com/posts'
response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    df = pd.DataFrame(data)
    print(df[['userId', 'id', 'title']].head())
else:
    print('请求失败')

   userId  id                   title
0       1   1  sunt aut facere repellat
1       1   2  qui est esse
2       1   3  ea molestias quasi exercitationem
3       1   4  eum et est occaecati
4       1   5  nesciunt quas odio


In [ ]:
import pandas as pd

# 练习：把下面 DataFrame 分别导出为 CSV 和 JSON
#       验证 orient='records' 和 force_ascii=False
df = pd.DataFrame(
    {'姓名': ['张三', '李四'], '年龄': [25, 30]}
)

# 学员代码区
pass

In [12]:
# 参考答案
import pandas as pd
import os

df = pd.DataFrame(
    {'姓名': ['张三', '李四'], '年龄': [25, 30]}
)

# 导出 CSV
df.to_csv('result.csv', index=False, encoding='utf-8')
print("=== CSV 文件内容 ===")
print(open('result.csv', encoding='utf-8').read())

# 导出 JSON
json_str = df.to_json(
    orient='records', force_ascii=False
)
print("=== JSON 字符串 ===")
print(json_str)

os.remove('result.csv')

=== CSV 文件内容 ===
姓名,年龄
张三,25
李四,30

=== JSON 字符串 ===
[{"姓名":"张三","年龄":25},{"姓名":"李四","年龄":30}]


## 总结
- `read_csv` / `read_excel`：注意 `encoding`、`skiprows`
- `read_json`：`orient` 要与 JSON 结构对应
- `read_sql`：配合 `with` 自动关闭连接
- `requests.get` 取 API 数据 → `pd.DataFrame(...)`
- 导出：`to_csv`、`to_json`、`to_sql` 记得关索引、设编码
本日知识结构图：
```
外部数据 ──▶ read_csv / read_excel
          ──▶ read_json / read_sql
          ──▶ requests.get → DataFrame

DataFrame ──▶ 清洗 ──▶ to_csv / to_json / to_sql
```

## ⭐ 挑战题
综合运用今天所有知识点：从 API 拉取数据，
清洗（处理缺失值），再导出为 CSV 文件。

In [ ]:
import requests
import pandas as pd

# 挑战题：从拉取 /users，清洗缺失的
#       'email' 字段（用 'unknown' 填充），
#       然后导出为 'users_clean.csv'
url = 'https://jsonplaceholder.typicode.com/users'

# 学员代码区
pass

In [13]:
# 参考答案
import requests
import pandas as pd
import os

url = 'https://jsonplaceholder.typicode.com/users'

# 1. 拉取
response = requests.get(url)
df = pd.DataFrame(response.json())

# 2. 清洗：email 缺失用 'unknown' 填充
if 'email' in df.columns:
    df['email'] = df['email'].fillna('unknown')

# 3. 导出 CSV
df.to_csv('users_clean.csv', index=False, encoding='utf-8')

print("=== 拉取、清洗、导出完成 ===")
print(df[['id', 'name', 'email']].head(3))

os.remove('users_clean.csv')

=== 拉取、清洗、导出完成 ===
   id     name                 email
0   1  Leanne Graham   Sincere@april.biz
1   2  Ervin Howell    Shanna@melissa.tv
2   3  Clementine Bauch  Nathan@yesenia.net
